# Article Tables (ICC 2026)
Reads the CSV files from the final runs and writes each table in LaTeX: prints it here and saves it to
`tabelas/<label>.tex` and, all together, to `tabelas.tex`.

| table | files |
|---|---|
| tab:util-cve5g | results_05_mlp.csv, results_05_cnn.csv |
| tab:util-nidd | results_07.csv |
| tab:calib | results_05_mlp.csv, results_07.csv |
| tab:eps | results_05_mlp.csv, results_07.csv (+ Opacus PRV accountant for the "centralized" row) |
| tab:mia | MLP: results_08_mia_cve5g_alwaysin.csv, results_09_art_cve5g_alwaysin.csv, results_08_mia_5gnidd_3sig.csv, results_09_art_5gnidd_3sig.csv; 1D-CNN: results_08_mia_{cve5g,5gnidd}_cnn.csv, results_09_art_{cve5g,5gnidd}_cnn.csv |
| tab:sia | results_08_mia_cve5g_alwaysin.csv, results_08_mia_5gnidd_3sig.csv |
| tab:custo | results_05_mlp.csv, results_05_cnn.csv, results_07.csv |

The numbers are formatted using English conventions (`DECIMAL = '.'`); `tab:mia`, which is included in the paper, has an English caption.

In [ ]:
import os
ARQ = {
    'cve_mlp':  'results_05_mlp.csv',
    'cve_cnn':  'results_05_cnn.csv',
    'nidd':     'results_07.csv',
    'cve_mia':  'results_08_mia_cve5g_alwaysin.csv',
    'cve_art':  'results_09_art_cve5g_alwaysin.csv',
    'nidd_mia': 'results_08_mia_5gnidd_3sig.csv',
    'nidd_art': 'results_09_art_5gnidd_3sig.csv',
    # auditoria da 1D-CNN (08 e 09 com --model cnn)
    'cve_mia_cnn':  'results_08_mia_cve5g_cnn.csv',
    'cve_art_cnn':  'results_09_art_cve5g_cnn.csv',
    'nidd_mia_cnn': 'results_08_mia_5gnidd_cnn.csv',
    'nidd_art_cnn': 'results_09_art_5gnidd_cnn.csv',
}
# o ART é opcional: sem o arquivo, a coluna sai com --
OPCIONAIS = {'cve_art_cnn', 'nidd_art_cnn'}
try:
    from google.colab import files
    if not all(os.path.exists(f) for k, f in ARQ.items() if k not in OPCIONAIS):
        files.upload()
except ImportError:
    files = None
faltam = [f for f in ARQ.values() if not os.path.exists(f)]   # os opcionais podem faltar
print('faltam:', faltam if faltam else 'nenhum')

faltam: nenhum


In [ ]:
import math
import subprocess
import sys
import numpy as np
import pandas as pd

DECIMAL, MILHAR = '.', ','

EPOCAS, LOTE, DELTA = 20, 64, 1e-6


def num(v, nd):
    '''Número com nd casas, separador decimal e de milhar do artigo.'''
    s = f"{v:,.{nd}f}"
    return s.replace(',', '\0').replace('.', DECIMAL).replace('\0', MILHAR)


def pm(m, s, nd):
    return f"{num(m, nd)}$\\pm${num(s, nd)}"


def faixa(lo, hi, nd=None):
    '''Faixa "lo--hi"; com nd=None usa 1 casa para valores ≥ 10 e 2 casas abaixo disso.'''
    if nd is None:
        nd = 1 if lo >= 10 else 2
    return f"{num(lo, nd)}--{num(hi, nd)}"


def eps_txt(e, nd=2):
    return r'$\infty$' if np.isinf(e) else num(e, nd)


def lista(s):
    '''"174;20152;20152" → array([174., 20152., 20152.])'''
    return np.array(str(s).split(';'), dtype=float)


def sel(df, cfg):
    g = df[df['config'] == cfg]
    return g if len(g) else None


def media(df, cfg, col, esc=1.0):
    g = sel(df, cfg)
    return None if g is None else float(esc * g[col].astype(float).mean())


def desvio(df, cfg, col, esc=1.0):
    '''Desvio padrão amostral (ddof=1) entre as sementes.'''
    g = sel(df, cfg)
    return None if g is None else float(esc * g[col].astype(float).std(ddof=1))


def tabela(linhas):
    return '\n'.join(linhas) + '\n'


SAIDA = {}


def salvar(rotulo, tex):
    os.makedirs('tabelas', exist_ok=True)
    nome = rotulo.replace('tab:', '')
    with open(f'tabelas/{nome}.tex', 'w') as f:
        f.write(tex)
    SAIDA[rotulo] = tex
    print(tex)


ROT_SIGMA = lambda s: r'$\sigma$=' + (num(s, 1) if s % 1 else f'{s:g}')
ROT_EPS = lambda e: r'$\varepsilon\le$' + f'{e:g}'

In [ ]:
mlp = pd.read_csv(ARQ['cve_mlp'])
cnn = pd.read_csv(ARQ['cve_cnn'])
nidd = pd.read_csv(ARQ['nidd'])
UTIL = {'CVE5G':   {'mlp': mlp, 'cnn': cnn},
        '5G-NIDD': {'mlp': nidd[nidd['model'] == 'mlp'], 'cnn': nidd[nidd['model'] == 'cnn']}}

ATAQUES_ART = ['art_rule', 'art_rf', 'art_nn_loss']


def ler_art(caminho):
    a = pd.read_csv(caminho)
    a = a[a['attack'].isin(ATAQUES_ART)].copy()
    for c in ('k', 'auc', 'n_per_side'):
        a[c] = pd.to_numeric(a[c])
    return a


def ler_art_opc(chave):
    return ler_art(ARQ[chave]) if os.path.exists(ARQ[chave]) else None


MIA = {('CVE5G', 'mlp'):   (pd.read_csv(ARQ['cve_mia']), ler_art(ARQ['cve_art'])),
       ('5G-NIDD', 'mlp'): (pd.read_csv(ARQ['nidd_mia']), ler_art(ARQ['nidd_art'])),
       ('CVE5G', 'cnn'):   (pd.read_csv(ARQ['cve_mia_cnn']), ler_art_opc('cve_art_cnn')),
       ('5G-NIDD', 'cnn'): (pd.read_csv(ARQ['nidd_mia_cnn']), ler_art_opc('nidd_art_cnn'))}
for ds, d in UTIL.items():
    print(ds, {m: sorted(df['config'].unique()) for m, df in d.items()})

CVE5G {'mlp': ['Central', 'Central+clip (s=0)', 'FL', 'FL+DP eps<=1', 'FL+DP eps<=2', 'FL+DP s=0.5', 'FL+DP s=1', 'FL+DP s=2', 'FL+clip (s=0)'], 'cnn': ['Central', 'Central+clip (s=0)', 'FL', 'FL+DP eps<=1', 'FL+DP eps<=2', 'FL+DP s=0.5', 'FL+DP s=1', 'FL+DP s=2', 'FL+clip (s=0)']}
5G-NIDD {'mlp': ['Central', 'Central+clip (s=0)', 'FL', 'FL+DP eps<=1', 'FL+DP eps<=2', 'FL+DP s=0.5', 'FL+DP s=1', 'FL+DP s=2', 'FL+clip (s=0)'], 'cnn': ['Central', 'Central+clip (s=0)', 'FL', 'FL+DP eps<=1', 'FL+DP eps<=2', 'FL+DP s=0.5', 'FL+DP s=1', 'FL+DP s=2', 'FL+clip (s=0)']}


## tab:util-cve5g — CVE5G

In [ ]:
CFG_UTIL = [('Central', 'Central'),
            ('Central+clip (s=0)', r'Central+\textit{clip}'),
            ('FL', 'FL'),
            ('FL+clip (s=0)', r'FL+\textit{clip}'),
            ('FL+DP s=0.5', 'FL+DP ' + ROT_SIGMA(0.5)),
            ('FL+DP s=1', 'FL+DP ' + ROT_SIGMA(1)),
            ('FL+DP s=2', 'FL+DP ' + ROT_SIGMA(2)),
            ('FL+DP eps<=1', 'FL+DP ' + ROT_EPS(1)),
            ('FL+DP eps<=2', 'FL+DP ' + ROT_EPS(2))]


def celulas_util(df, cfg):
    '''"F1 & FAR(%)" (média ± desvio) ou "—" ocupando as duas colunas.'''
    if sel(df, cfg) is None:
        return r'\multicolumn{2}{c}{—}'
    return (pm(media(df, cfg, 'macro_f1'), desvio(df, cfg, 'macro_f1'), 3) + ' & '
            + pm(media(df, cfg, 'far', 100), desvio(df, cfg, 'far', 100), 1))


def tab_util_cve5g():
    d = UTIL['CVE5G']
    # DR por configuração (média das sementes), para a frase da legenda
    drs = [media(df, k, 'det_rate') for df in d.values() for k, _ in CFG_UTIL
           if sel(df, k) is not None]
    if all(num(v, 3) == num(1, 3) for v in drs):
        frase_dr = f'A taxa de detecção (DR) é {num(1, 3)} em todas as configurações.'
    else:
        frase_dr = (f'A taxa de detecção (DR) fica entre {num(min(drs), 3)} e '
                    f'{num(max(drs), 3)}.')
    n_sem = d['mlp'].groupby('config')['seed'].nunique().max()
    L = [r'\begin{table}[htbp]',
         r'\caption{Detecção no CVE5G (média $\pm$ desvio de ' + f'{n_sem}' + r' sementes). '
         + frase_dr + r' $\varepsilon\le E$: $\sigma$'
         + '\nescolhido por cliente para que todos tenham $\\varepsilon\\le E$.}',
         r'\label{tab:util-cve5g}',
         r'\centering',
         r'\footnotesize',
         r'\setlength{\tabcolsep}{2.5pt}',
         r'\begin{tabular}{lcccc}',
         r'\hline',
         r'& \multicolumn{2}{c}{\textbf{MLP}} & \multicolumn{2}{c}{\textbf{1D-CNN}} \\',
         r'\textbf{Configuração} & \textbf{F1 macro} & \textbf{FAR (\%)} & \textbf{F1 macro} & \textbf{FAR (\%)} \\',
         r'\hline']
    for k, rot in CFG_UTIL:
        L.append(f"{rot:<28} & {celulas_util(d['mlp'], k)} & {celulas_util(d['cnn'], k)} \\\\")
    L += [r'\hline', r'\end{tabular}', r'\end{table}']
    return tabela(L)


salvar('tab:util-cve5g', tab_util_cve5g())

\begin{table}[htbp]
\caption{Detecção no CVE5G (média $\pm$ desvio de 5 sementes). A taxa de detecção (DR) é 1.000 em todas as configurações. $\varepsilon\le E$: $\sigma$
escolhido por cliente para que todos tenham $\varepsilon\le E$.}
\label{tab:util-cve5g}
\centering
\footnotesize
\setlength{\tabcolsep}{2.5pt}
\begin{tabular}{lcccc}
\hline
& \multicolumn{2}{c}{\textbf{MLP}} & \multicolumn{2}{c}{\textbf{1D-CNN}} \\
\textbf{Configuração} & \textbf{F1 macro} & \textbf{FAR (\%)} & \textbf{F1 macro} & \textbf{FAR (\%)} \\
\hline
Central                      & 0.999$\pm$0.001 & 0.1$\pm$0.2 & 1.000$\pm$0.000 & 0.0$\pm$0.0 \\
Central+\textit{clip}        & 1.000$\pm$0.001 & 0.0$\pm$0.0 & 1.000$\pm$0.000 & 0.0$\pm$0.0 \\
FL                           & 1.000$\pm$0.000 & 0.2$\pm$0.3 & 1.000$\pm$0.000 & 0.1$\pm$0.2 \\
FL+\textit{clip}             & 0.999$\pm$0.001 & 0.0$\pm$0.0 & 1.000$\pm$0.000 & 0.0$\pm$0.0 \\
FL+DP $\sigma$=0.5           & 0.993$\pm$0.001 & 3.8$\pm$0.4 & 0.992$\pm$0.004 & 4.8

## tab:util-nidd — 5G-NIDD

In [ ]:
REFERENCIAS = [('RF', 'Random Forest'), ('LogReg', 'Reg. logística')]
CFG_REF = 'Central (referência)'


def tab_util_nidd():
    d = UTIL['5G-NIDD']
    n_sem = d['mlp'].groupby('config')['seed'].nunique().max()
    L = [r'\begin{table}[htbp]',
         r'\caption{Detecção no 5G-NIDD (média $\pm$ desvio de ' + f'{n_sem}' + r' sementes), com a DR da',
         r'MLP e referências centralizadas sem privacidade.}',
         r'\label{tab:util-nidd}',
         r'\centering',
         r'\footnotesize',
         r'\setlength{\tabcolsep}{2pt}',
         r'\begin{tabular}{lccccc}',
         r'\hline',
         r'& \multicolumn{3}{c}{\textbf{MLP}} & \multicolumn{2}{c}{\textbf{1D-CNN}} \\',
         r'\textbf{Configuração} & \textbf{F1 macro} & \textbf{FAR (\%)} & \textbf{DR} &',
         r'\textbf{F1 macro} & \textbf{FAR (\%)} \\',
         r'\hline']
    for k, rot in CFG_UTIL:
        m = d['mlp']
        if sel(m, k) is None:
            mlp_txt = r'\multicolumn{3}{c}{—}'
        else:
            mlp_txt = celulas_util(m, k) + ' & ' + num(media(m, k, 'det_rate'), 3)
        L.append(f"{rot:<28} & {mlp_txt} & {celulas_util(d['cnn'], k)} \\\\")
    L.append(r'\hline')
    for modelo, rot in REFERENCIAS:
        r = nidd[nidd['model'] == modelo]
        if sel(r, CFG_REF) is None:
            continue
        L.append(f"{rot:<28} & {celulas_util(r, CFG_REF)} & {num(media(r, CFG_REF, 'det_rate'), 3)}"
                 r" & \multicolumn{2}{c}{—} \\")
    L += [r'\hline', r'\end{tabular}', r'\end{table}']
    return tabela(L)


salvar('tab:util-nidd', tab_util_nidd())

\begin{table}[htbp]
\caption{Detecção no 5G-NIDD (média $\pm$ desvio de 5 sementes), com a DR da
MLP e referências centralizadas sem privacidade.}
\label{tab:util-nidd}
\centering
\footnotesize
\setlength{\tabcolsep}{2pt}
\begin{tabular}{lccccc}
\hline
& \multicolumn{3}{c}{\textbf{MLP}} & \multicolumn{2}{c}{\textbf{1D-CNN}} \\
\textbf{Configuração} & \textbf{F1 macro} & \textbf{FAR (\%)} & \textbf{DR} &
\textbf{F1 macro} & \textbf{FAR (\%)} \\
\hline
Central                      & 0.885$\pm$0.001 & 0.2$\pm$0.0 & 0.885 & 0.883$\pm$0.001 & 0.2$\pm$0.1 \\
Central+\textit{clip}        & 0.885$\pm$0.000 & 0.2$\pm$0.0 & 0.884 & 0.883$\pm$0.001 & 0.5$\pm$0.3 \\
FL                           & 0.873$\pm$0.022 & 0.5$\pm$0.3 & 0.872 & 0.835$\pm$0.001 & 0.7$\pm$0.4 \\
FL+\textit{clip}             & 0.856$\pm$0.039 & 7.1$\pm$14.6 & 0.879 & 0.844$\pm$0.021 & 9.1$\pm$18.0 \\
FL+DP $\sigma$=0.5           & 0.815$\pm$0.016 & 5.4$\pm$4.9 & 0.822 & 0.782$\pm$0.033 & 5.3$\pm$1.3 \\
FL+DP $\sigma$=1       

## tab:calib — calibrated decision threshold (MLP)


In [ ]:
CFG_CAL = [('FL', 'FL'),
           ('FL+DP s=0.5', ROT_SIGMA(0.5)),
           ('FL+DP s=1', ROT_SIGMA(1)),
           ('FL+DP s=2', ROT_SIGMA(2)),
           ('FL+DP eps<=2', ROT_EPS(2))]


def antes_depois(df, cfg, col, esc, nd):
    return f"{num(media(df, cfg, col, esc), nd)}/{num(media(df, cfg, col + '_cal', esc), nd)}"


def tab_calib():
    L = [r'\begin{table}[htbp]',
         r'\caption{Efeito do limiar de decisão calibrado na validação (MLP; média de 5',
         r"sementes). Em cada célula, ``Arg./Cal.'': decisão pela classe mais provável /",
         r'pelo limiar calibrado (índice de Youden). F1: F1 macro. As linhas com $\sigma$ e $\varepsilon$ são FL+DP.}',
         r'\label{tab:calib}',
         r'\centering',
         r'\footnotesize',
         r'\setlength{\tabcolsep}{0.9pt}',
         r'\begin{tabular}{lccc|ccc}',
         r'\hline',
         r'& \multicolumn{3}{c|}{\textbf{CVE5G}} & \multicolumn{3}{c}{\textbf{5G-NIDD}} \\',
         r'\textbf{Config.} & \textbf{F1} & \textbf{FAR (\%)} & \textbf{DR} & \textbf{F1} & \textbf{FAR (\%)} & \textbf{DR} \\',
         r'\hline']
    for k, rot in CFG_CAL:
        cel = []
        for ds in ('CVE5G', '5G-NIDD'):
            df = UTIL[ds]['mlp']
            if sel(df, k) is None:
                cel += ['—'] * 3
                continue
            cel += [antes_depois(df, k, 'macro_f1', 1, 3),
                    antes_depois(df, k, 'far', 100, 1),
                    # a coluna calibrada da DR chama det_rate_cal
                    f"{num(media(df, k, 'det_rate'), 3)}/{num(media(df, k, 'det_rate_cal'), 3)}"]
        L.append(f"{rot:<24} & " + ' & '.join(cel) + r' \\')
    L += [r'\hline', r'\end{tabular}', r'\end{table}']
    return tabela(L)


salvar('tab:calib', tab_calib())

\begin{table}[htbp]
\caption{Efeito do limiar de decisão calibrado na validação (MLP; média de 5
sementes). Em cada célula, ``Arg./Cal.'': decisão pela classe mais provável /
pelo limiar calibrado (índice de Youden). F1: F1 macro. As linhas com $\sigma$ e $\varepsilon$ são FL+DP.}
\label{tab:calib}
\centering
\footnotesize
\setlength{\tabcolsep}{0.9pt}
\begin{tabular}{lccc|ccc}
\hline
& \multicolumn{3}{c|}{\textbf{CVE5G}} & \multicolumn{3}{c}{\textbf{5G-NIDD}} \\
\textbf{Config.} & \textbf{F1} & \textbf{FAR (\%)} & \textbf{DR} & \textbf{F1} & \textbf{FAR (\%)} & \textbf{DR} \\
\hline
FL                       & 1.000/0.999 & 0.2/0.3 & 1.000/1.000 & 0.873/0.883 & 0.5/0.6 & 0.872/0.884 \\
$\sigma$=0.5             & 0.993/0.908 & 3.8/0.2 & 1.000/0.995 & 0.815/0.824 & 5.4/2.8 & 0.822/0.822 \\
$\sigma$=1               & 0.981/0.798 & 10.5/0.5 & 1.000/0.991 & 0.778/0.817 & 8.9/4.7 & 0.788/0.821 \\
$\sigma$=2               & 0.954/0.787 & 23.5/1.6 & 1.000/0.989 & 0.763/0.811 & 10.1/6.4 & 0.774

## tab:eps — privacy budget ε per client

The per-client ε values are obtained from the `eps_clients`/`eps`/`eps_median` columns, the σ values from the ε≤2 configuration in `sigma_clients`, and the number of flows from `n_clients_train`. The "centralized" row (the ε that centralized training would incur with the same σ) is not available in any CSV file: it is computed here using the same Opacus PRV accountant as the scripts (install `opacus` if missing).

In [ ]:
try:
    from opacus.accountants import PRVAccountant
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opacus'], check=False)
    try:
        from opacus.accountants import PRVAccountant
    except ImportError:
        PRVAccountant = None
        print('opacus indisponível: a linha "central" da tab:eps sai com —')

SIGMAS = [0.5, 1, 2]
CFG_SIG = {s: f'FL+DP s={s:g}' for s in SIGMAS}
CFG_ALVO_EPS = 'FL+DP eps<=2'


def eps_central(n, sigma):
    '''Mesma conta do epsilon_for() dos scripts: Poisson com q = 1/passos_por_época.'''
    if PRVAccountant is None:
        return None
    spe = math.ceil(n / LOTE)
    acc = PRVAccountant()
    acc.history = [(sigma, 1.0 / spe, spe * EPOCAS)]
    return float(acc.get_epsilon(DELTA))


def linhas_cve5g():
    df = UTIL['CVE5G']['mlp']
    tam = lista(sel(df, 'FL')['n_clients_train'].iloc[0]).astype(int)
    eps = {s: lista(sel(df, CFG_SIG[s])['eps_clients'].iloc[0]) for s in SIGMAS}
    sig = lista(sel(df, CFG_ALVO_EPS)['sigma_clients'].iloc[0])
    for s in SIGMAS:
        assert sel(df, CFG_SIG[s])['eps_clients'].nunique() == 1, 'ε por cliente muda entre sementes?'
    grupos = {}
    for i, n in enumerate(tam):
        grupos.setdefault(n, []).append(i)
    L = []
    for j, (n, idx) in enumerate(grupos.items()):
        nome = ', '.join(f's{i + 1}' for i in idx)
        i = idx[0]
        L.append(f"{'CVE5G' if j == 0 else '':<8}& {nome:<15} & {num(n, 0):<6} & "
                 + ' & '.join(num(eps[s][i], 2) for s in SIGMAS) + f" & {num(sig[i], 2)} \\\\")
    n_c = int(tam.sum())
    ec = [eps_central(n_c, s) for s in SIGMAS]
    L.append(f"{'':<8}& {'central':<15} & {num(n_c, 0):<6} & "
             + ' & '.join('—' if e is None else num(e, 2) for e in ec) + r' & — \\')
    return L


def linhas_nidd():
    df = UTIL['5G-NIDD']['mlp']
    alvo = sel(df, CFG_ALVO_EPS)
    sig_pior = [lista(s).max() for s in alvo['sigma_clients']]        # cliente menor = maior σ
    sig_med = [np.median(lista(s)) for s in alvo['sigma_clients']]
    L = []
    for j, (rot, col, sig) in enumerate([('pior', 'eps', sig_pior),
                                          ('mediana', 'eps_median', sig_med)]):
        cel = []
        for s in SIGMAS:
            v = sel(df, CFG_SIG[s])[col].astype(float)
            cel.append(faixa(v.min(), v.max()))
        L.append(f"{'5G-NIDD' if j == 0 else '':<8}& {rot:<15} & {'—':<6} & " + ' & '.join(cel)
                 + f" & {faixa(min(sig), max(sig), 2)} \\\\")
    ns = (sel(df, 'FL').drop_duplicates('seed')['n_clients_train']
          .map(lambda s: int(lista(s).sum())).tolist())
    cel = []
    for s in SIGMAS:
        e = [eps_central(n, s) for n in ns]
        cel.append('—' if e[0] is None else faixa(min(e), max(e), 2))
    aprox = num(round(float(np.median(ns)), -3), 0)
    L.append(f"{'':<8}& {'central':<15} & $\\approx${aprox} & " + ' & '.join(cel) + r' & — \\')
    return L, ns


def tab_eps():
    l_nidd, ns = linhas_nidd()
    cab_sig = ' & '.join(r'$\boldsymbol{\sigma}$\textbf{=' + (num(s, 1) if s % 1 else f'{s:g}') + '}'
                         for s in SIGMAS)
    L = ['% σ por cliente com ε alvo e faixa de fluxos do treino central do 5G-NIDD: '
         f'{num(min(ns), 0)} a {num(max(ns), 0)}.',
         r'\begin{table}[htbp]',
         r'\caption{Orçamento $\varepsilon$ por cliente ($\delta=10^{-6}$; 10 rodadas',
         r'$\times$ 2 épocas locais; lote de 64; contador PRV). No 5G-NIDD, a partição',
         r'muda a cada semente; mostramos a faixa entre as sementes. O s1 só tem tráfego benigno. Última coluna:',
         r'$\sigma$ que cada cliente usa na configuração $\varepsilon\le 2$. Linha',
         r"``central'': $\varepsilon$ que o treino centralizado teria com o mesmo $\sigma$",
         r'(referência; o Central+DP não foi avaliado).}',
         r'\label{tab:eps}',
         r'\centering',
         r'\footnotesize',
         r'\setlength{\tabcolsep}{2pt}',
         r'\begin{tabular}{llrcccc}',
         r'\hline',
         r'& \textbf{Cliente} & \textbf{Fluxos} & ' + cab_sig + ' &',
         r'$\boldsymbol{\sigma}$\textbf{,} $\boldsymbol{\varepsilon\le2}$ \\',
         r'\hline']
    L += linhas_cve5g() + [r'\hline'] + l_nidd
    L += [r'\hline', r'\end{tabular}', r'\end{table}']
    return tabela(L)


salvar('tab:eps', tab_eps())

% σ por cliente com ε alvo e faixa de fluxos do treino central do 5G-NIDD: 60,766 a 72,296.
\begin{table}[htbp]
\caption{Orçamento $\varepsilon$ por cliente ($\delta=10^{-6}$; 10 rodadas
$\times$ 2 épocas locais; lote de 64; contador PRV). No 5G-NIDD, a partição
muda a cada semente; mostramos a faixa entre as sementes. O s1 só tem tráfego benigno. Última coluna:
$\sigma$ que cada cliente usa na configuração $\varepsilon\le 2$. Linha
``central'': $\varepsilon$ que o treino centralizado teria com o mesmo $\sigma$
(referência; o Central+DP não foi avaliado).}
\label{tab:eps}
\centering
\footnotesize
\setlength{\tabcolsep}{2pt}
\begin{tabular}{llrcccc}
\hline
& \textbf{Cliente} & \textbf{Fluxos} & $\boldsymbol{\sigma}$\textbf{=0.5} & $\boldsymbol{\sigma}$\textbf{=1} & $\boldsymbol{\sigma}$\textbf{=2} &
$\boldsymbol{\sigma}$\textbf{,} $\boldsymbol{\varepsilon\le2}$ \\
\hline
CVE5G   & s1              & 174    & 72.32 & 20.83 & 7.53 & 5.98 \\
        & s2, s3          & 20,152 & 12.66 & 1.53

## tab:mia — canary audit (MLP and 1D-CNN)

Columns for each model: loss-based attack, online LiRA (AUC and TPR@1%), server-side LiRA on model updates (`srv_lira`), best ART attack (highest mean AUC across partitions among the three attacks), and empirical ε (highest `eps_lb` across the attacks from script 08, on the canaries). The audit ε does not depend on the model (only on σ and the data) and is therefore reported once. Before generating the table, the cell checks whether the MLP and CNN were audited using the same protocol (same number of audited flows and canaries) and issues a warning otherwise.

In [ ]:
CFG_MIA = [('Central', 'C'),
           ('FL', 'FL'),
           ('FL+DP s=0.5', 'FL+DP ' + ROT_SIGMA(0.5)),
           ('FL+DP s=1', 'FL+DP ' + ROT_SIGMA(1)),
           ('FL+DP s=2', 'FL+DP ' + ROT_SIGMA(2)),
           ('Overfit (controle +)', 'PC')]
CFG_PC = 'Overfit (controle +)'
MODELOS_MIA = [('mlp', 'MLP'), ('cnn', '1D-CNN')]


def valor_mia(r, cfg, subset, ataque, col='auc_mean'):
    g = r[(r['config'] == cfg) & (r['subset'] == subset) & (r['attack'] == ataque)]
    return None if g.empty else float(g[col].iloc[0])


def art_melhor(a, cfg, subset='canary'):
    if a is None:
        return None
    g = a[(a['config'] == cfg) & (a['subset'] == subset)]
    return None if g.empty else float(g.groupby('attack')['auc'].mean().max())


def acaso_art(a, subset='canary'):
    g = a[a['subset'] == subset]
    n, k = int(g['n_per_side'].iloc[0]), g['k'].nunique()
    return n, 0.5 + 1.96 * math.sqrt((2 * n + 1) / (12 * n * n)) / math.sqrt(k)


def valores_modelo(r, a, cfg):
    '''[perda, LiRA, TPR@1%, servidor, ART, ε̂] de uma configuração (None se faltar).'''
    g = r[r['config'] == cfg]
    if g.empty:
        return [None] * 6
    can = g[g['subset'] == 'canary']
    return [valor_mia(r, cfg, 'canary', 'loss'),
            valor_mia(r, cfg, 'canary', 'lira_on'),
            valor_mia(r, cfg, 'canary', 'lira_on', 'tpr1'),
            valor_mia(r, cfg, 'canary', 'srv_lira'),
            art_melhor(a, cfg),
            float(can['eps_lb'].astype(float).max())]


def conferir_auditorias():
    '''A MLP e a CNN precisam ter o mesmo protocolo: mesmos canários e mesmo ε audit.'''
    for ds in ('CVE5G', '5G-NIDD'):
        r_m, r_c = MIA[(ds, 'mlp')][0], MIA[(ds, 'cnn')][0]
        n_m = r_m[r_m['subset'] == 'canary']['n_per_target'].iloc[0]
        n_c = r_c[r_c['subset'] == 'canary']['n_per_target'].iloc[0]
        a_m = r_m[r_m['subset'] == 'all']['n_per_target'].iloc[0]
        a_c = r_c[r_c['subset'] == 'all']['n_per_target'].iloc[0]
        if (n_m, a_m) != (n_c, a_c):
            print(f'ATENÇÃO {ds}: MLP audita {a_m} fluxos e {n_m} canários; CNN audita {a_c} '
                  f'fluxos e {n_c} canários. Protocolos diferentes (ex.: --dup-policy).')
        e_m = r_m.groupby('config')['eps_audit'].first()
        e_c = r_c.groupby('config')['eps_audit'].first()
        comum = e_m.index.intersection(e_c.index)
        if not np.allclose(e_m[comum].astype(float), e_c[comum].astype(float), equal_nan=True):
            print(f'ATENÇÃO {ds}: ε audit diferente entre MLP e CNN.')


def tab_mia():
    conferir_auditorias()
    L = [r'\begin{table*}[t]',
         r'\caption{Membership inference audit on canaries (AUC; chance = 0.5). C: centralized;',
         r'PC: positive control (centralized MLP without DP, trained to memorize). Loss and',
         r'LiRA: attacks on the final model; TPR@1\%: LiRA at 1\% FPR (chance = 0.01);',
         r'Server: LiRA on the local updates; ART: best attack of the ART library;',
         r'$\varepsilon$ audit: budget of the audit models (the same for both models);',
         r'$\hat{\varepsilon}$: empirical lower bound on $\varepsilon$. Bold: lowest attack success',
         r'per dataset and model among C, FL, and FL+DP.']
    lims = []
    for ds in ('CVE5G', '5G-NIDD'):
        vs = {(n, round(l, 2)) for n, l in (acaso_art(MIA[(ds, mod)][1])
              for mod, _ in MODELOS_MIA if MIA[(ds, mod)][1] is not None)}
        if len(vs) > 1:
            print(f'aviso: {ds} tem números de canários diferentes no ART entre os modelos', vs)
        if vs:
            n, lim = max(vs, key=lambda t: t[1])
            lims.append(f'$\\approx${num(lim, 2)} in {ds} ({n} canaries per side)')
    if lims:
        L.append('ART values up to ' + ' and '.join(lims) + ' are consistent with chance.}')
    else:
        L[-1] += '}'
    L += [r'\label{tab:mia}',
          r'\centering',
          r'\footnotesize',
          r'\setlength{\tabcolsep}{2.5pt}',
          r'\begin{tabular}{llc|cccccc|cccccc}',
          r'\hline',
          r'& & & \multicolumn{6}{c|}{\textbf{MLP}} & \multicolumn{6}{c}{\textbf{1D-CNN}} \\',
          r'\textbf{Dataset} & \textbf{IDS} & $\boldsymbol{\varepsilon}$ \textbf{audit}'
          + r' & \textbf{Loss} & \textbf{LiRA} & \textbf{TPR@1\%} & \textbf{Server} & \textbf{ART}'
          + r' & $\boldsymbol{\hat{\varepsilon}}$' * 1
          + r' & \textbf{Loss} & \textbf{LiRA} & \textbf{TPR@1\%} & \textbf{Server} & \textbf{ART}'
          + r' & $\boldsymbol{\hat{\varepsilon}}$ \\',
          r'\hline']
    nd = [3, 3, 3, 3, 3, 2]
    for ds in ('CVE5G', '5G-NIDD'):
        r_m = MIA[(ds, 'mlp')][0]
        cfgs = [(k, rot) for k, rot in CFG_MIA if not r_m[r_m['config'] == k].empty]
        vals = {mod: {k: valores_modelo(MIA[(ds, mod)][0], MIA[(ds, mod)][1], k) for k, _ in cfgs}
                for mod, _ in MODELOS_MIA}
        eps = {k: float(r_m[r_m['config'] == k]['eps_audit'].iloc[0]) for k, _ in cfgs}
        sem_pc = [k for k, _ in cfgs if k != CFG_PC]
        best = {}
        for mod, _ in MODELOS_MIA:
            for c in range(6):
                vs = [round(vals[mod][k][c], nd[c]) for k in sem_pc if vals[mod][k][c] is not None]
                best[(mod, c)] = min(vs) if vs else None
        eps_fin = [eps[k] for k in sem_pc if not np.isinf(eps[k])]
        eps_best = min(eps_fin) if eps_fin else None
        for j, (k, rot) in enumerate(cfgs):
            e = eps[k]
            cel = [r'\textbf{' + eps_txt(e) + '}' if eps_best is not None and e == eps_best
                   else eps_txt(e)]
            for mod, _ in MODELOS_MIA:
                for c, v in enumerate(vals[mod][k]):
                    if v is None:
                        cel.append('--')
                        continue
                    t = num(v, nd[c])
                    if k != CFG_PC and best[(mod, c)] is not None and round(v, nd[c]) == best[(mod, c)]:
                        t = r'\textbf{' + t + '}'
                    cel.append(t)
            L.append(f"{ds if j == 0 else '':<8}& {rot:<19} & " + ' & '.join(cel) + r' \\')
        L.append(r'\hline')
    L += [r'\end{tabular}', r'\end{table*}']
    return tabela(L)


salvar('tab:mia', tab_mia())


\begin{table*}[t]
\caption{Membership inference audit on canaries (AUC; chance = 0.5). C: centralized;
PC: positive control (centralized MLP without DP, trained to memorize). Loss and
LiRA: attacks on the final model; TPR@1\%: LiRA at 1\% FPR (chance = 0.01);
Server: LiRA on the local updates; ART: best attack of the ART library;
$\varepsilon$ audit: budget of the audit models (the same for both models);
$\hat{\varepsilon}$: empirical lower bound on $\varepsilon$. Bold: lowest attack success
per dataset and model among C, FL, and FL+DP.
ART values up to $\approx$0.58 in CVE5G (23 canaries per side) and $\approx$0.53 in 5G-NIDD (155 canaries per side) are consistent with chance.}
\label{tab:mia}
\centering
\footnotesize
\setlength{\tabcolsep}{2.5pt}
\begin{tabular}{llc|cccccc|cccccc}
\hline
& & & \multicolumn{6}{c|}{\textbf{MLP}} & \multicolumn{6}{c}{\textbf{1D-CNN}} \\
\textbf{Dataset} & \textbf{IDS} & $\boldsymbol{\varepsilon}$ \textbf{audit} & \textbf{Loss} & \textbf{LiRA} & \textbf{

## tab:sia — source inference

In [ ]:
CFG_SIA = [('FL', 'FL'),
           ('FL+DP s=0.5', 'FL+DP ' + ROT_SIGMA(0.5)),
           ('FL+DP s=1', 'FL+DP ' + ROT_SIGMA(1)),
           ('FL+DP s=2', 'FL+DP ' + ROT_SIGMA(2))]


def tab_sia():
    L = [r'\begin{table}[htbp]',
         r'\caption{\textit{Source inference}: acerto do servidor ao identificar o',
         r'cliente de origem de um fluxo membro, comparado com o acerto obtido apenas',
         r'pela classe do fluxo.}',
         r'\label{tab:sia}',
         r'\centering',
         r'\footnotesize',
         r'\setlength{\tabcolsep}{3pt}',
         r'\begin{tabular}{llcccc}',
         r'\hline',
         r'& & \multicolumn{2}{c}{\textbf{Servidor}} & \multicolumn{2}{c}{\textbf{Só pela classe}} \\',
         r'\textbf{Dataset} & \textbf{Configuração} & \textbf{Todos} & \textbf{Benigno} & \textbf{Todos} & \textbf{Benigno} \\',
         r'\hline']
    acaso = []
    for ds in ('CVE5G', '5G-NIDD'):
        r = MIA[(ds, 'mlp')][0]
        s = r[r['attack'] == 'source_inference']
        for j, (k, rot) in enumerate(CFG_SIA):
            g = s[s['config'] == k].set_index('subset')
            if g.empty:
                continue
            cel = [num(g.loc['all', 'sia_acc'], 3), num(g.loc['benign', 'sia_acc'], 3),
                   num(g.loc['all', 'sia_label_prior'], 3), num(g.loc['benign', 'sia_label_prior'], 3)]
            L.append(f"{ds if j == 0 else '':<8}& {rot:<18} & " + ' & '.join(cel) + r' \\')
        p = float(s['sia_random'].iloc[0])
        acaso.append(f'{num(p, 3)} ({ds}, {round(1 / p)} clientes)')
    L += [r'\hline',
          r'\multicolumn{6}{l}{Acaso: ' + ' e '.join(acaso) + '.}',
          r'\end{tabular}', r'\end{table}']
    return tabela(L)


salvar('tab:sia', tab_sia())

\begin{table}[htbp]
\caption{\textit{Source inference}: acerto do servidor ao identificar o
cliente de origem de um fluxo membro, comparado com o acerto obtido apenas
pela classe do fluxo.}
\label{tab:sia}
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{llcccc}
\hline
& & \multicolumn{2}{c}{\textbf{Servidor}} & \multicolumn{2}{c}{\textbf{Só pela classe}} \\
\textbf{Dataset} & \textbf{Configuração} & \textbf{Todos} & \textbf{Benigno} & \textbf{Todos} & \textbf{Benigno} \\
\hline
CVE5G   & FL                 & 0.503 & 0.332 & 0.544 & 0.351 \\
        & FL+DP $\sigma$=0.5 & 0.548 & 0.352 & 0.544 & 0.351 \\
        & FL+DP $\sigma$=1   & 0.513 & 0.351 & 0.544 & 0.351 \\
        & FL+DP $\sigma$=2   & 0.513 & 0.351 & 0.544 & 0.351 \\
5G-NIDD & FL                 & 0.292 & 0.281 & 0.318 & 0.319 \\
        & FL+DP $\sigma$=0.5 & 0.314 & 0.309 & 0.318 & 0.319 \\
        & FL+DP $\sigma$=1   & 0.314 & 0.310 & 0.318 & 0.319 \\
        & FL+DP $\sigma$=2   & 0.314 & 0.310 & 0

## tab:custo — MLP training cost

In [ ]:
CUSTO_DP = 'FL+DP s=1'
CFG_CUSTO = [('Central', 'Central'),
             ('FL', 'FL'),
             ('FL+clip (s=0)', r'FL+\textit{clip}'),
             (CUSTO_DP, 'FL+DP')]


def tab_custo():
    sig = float(sel(mlp, CUSTO_DP)['sigma'].iloc[0])
    L = [r'\begin{table}[htbp]',
         r'\caption{Custo de treinamento da MLP (média de 5 sementes; FL+DP com',
         f'$\\sigma={num(sig, 0) if sig % 1 == 0 else num(sig, 1)}$). Tempo do treino local em paralelo e somado entre os clientes,',
         r'sem a sobrecarga do simulador.}',
         r'\label{tab:custo}',
         r'\centering',
         r'\footnotesize',
         r'\setlength{\tabcolsep}{2.5pt}',
         r'\begin{tabular}{lcccc}',
         r'\hline',
         r'& \multicolumn{2}{c}{\textbf{CVE5G}} & \multicolumn{2}{c}{\textbf{5G-NIDD}} \\',
         r'\textbf{Configuração} & \textbf{Tempo (s)} & \textbf{kB/rodada} & \textbf{Tempo (s)} & \textbf{kB/rodada} \\',
         r'\hline']
    for k, rot in CFG_CUSTO:
        cel = []
        for ds in ('CVE5G', '5G-NIDD'):
            df = UTIL[ds]['mlp']
            par = media(df, k, 'client_parallel_s')
            if k.startswith('Central'):
                cel += [num(par, 1), '—']
            else:
                cel += [f"{num(par, 1)} / {num(media(df, k, 'client_train_s'), 1)}",
                        num(media(df, k, 'kb_per_round'), 1)]
        L.append(f"{rot:<16}& " + ' & '.join(cel) + r' \\')
    par = {m: ' / '.join(num(UTIL[ds][m]['n_params'].iloc[0], 0) for ds in ('CVE5G', '5G-NIDD'))
           for m in ('mlp', 'cnn')}
    L += [r'\hline',
          r'\multicolumn{5}{l}{FL: paralelo / soma. Parâmetros: MLP ' + par['mlp'] + r';}\\',
          r'\multicolumn{5}{l}{1D-CNN ' + par['cnn'] + '.}',
          r'\end{tabular}', r'\end{table}']
    return tabela(L)


salvar('tab:custo', tab_custo())

\begin{table}[htbp]
\caption{Custo de treinamento da MLP (média de 5 sementes; FL+DP com
$\sigma=1$). Tempo do treino local em paralelo e somado entre os clientes,
sem a sobrecarga do simulador.}
\label{tab:custo}
\centering
\footnotesize
\setlength{\tabcolsep}{2.5pt}
\begin{tabular}{lcccc}
\hline
& \multicolumn{2}{c}{\textbf{CVE5G}} & \multicolumn{2}{c}{\textbf{5G-NIDD}} \\
\textbf{Configuração} & \textbf{Tempo (s)} & \textbf{kB/rodada} & \textbf{Tempo (s)} & \textbf{kB/rodada} \\
\hline
Central         & 7.5 & — & 12.2 & — \\
FL              & 3.9 / 8.3 & 307.3 & 3.8 / 13.9 & 611.9 \\
FL+\textit{clip}& 9.9 / 20.7 & 307.3 & 13.2 / 51.1 & 611.9 \\
FL+DP           & 10.1 / 21.2 & 307.3 & 13.5 / 52.1 & 611.9 \\
\hline
\multicolumn{5}{l}{FL: paralelo / soma. Parâmetros: MLP 12,803 / 15,298;}\\
\multicolumn{5}{l}{1D-CNN 7,875 / 12,962.}
\end{tabular}
\end{table}



## Download



In [ ]:
ORDEM = ['tab:util-cve5g', 'tab:util-nidd', 'tab:calib', 'tab:eps', 'tab:mia', 'tab:sia',
         'tab:custo']
with open('tabelas.tex', 'w') as f:
    f.write('\\section{Tabelas}\n' + '\n\n'.join(SAIDA[k] for k in ORDEM if k in SAIDA))
print('salvo: tabelas.tex e tabelas/*.tex')
if files is not None:
    files.download('tabelas.tex')

salvo: tabelas.tex e tabelas/*.tex


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>